In [1]:
# %pip install scikit-learn
# %pip install xgboost
# %pip install shap
# %pip install matplotlib
# %pip install pyarrow

In [2]:
# -*- coding: utf-8 -*-
"""
02. 트랙 A 모델링 스크립트
--------------------------------
전제: 01_씬파일러_사전검증.py 실행 후 'is_thin' 플래그가 붙은
      data/sheet9_with_thin_flag.parquet 파일이 존재해야 함

구성
----
Part 1. 씬파일러군 내부 — 기준모델 vs 대안모델 비교 (AUC/PR-AUC)
Part 2. 전체 모집단 — 씬파일러더미 + 대안변수 교호항으로 H7(대체 vs 보완) 검증
Part 3. SHAP으로 대안변수 기여도 순위 산출
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap

pd.set_option("display.width", 120)
RANDOM_STATE = 42

TARGET = "PERF1"  # 90일 이상 연체 여부 (다른 PERF로 바꿔가며 반복 실행 가능)

# 씬파일러 정의 필드 — 모델 피처에서 반드시 제외 (zero-variance)
THIN_FIELDS = ["C1M210000", "C18233003", "C18233004", "C18233005", "L10220000"]

BASELINE_FEATURES = ["SCORE", "SCORE_6M"]
ALT_FEATURES = ["AP0910001", "AP0910002", "AL012G005", "AL012G011", "AL012G019"]


# ------------------------------------------------------------------
# 0. 데이터 로드
# ------------------------------------------------------------------
def load_data() -> pd.DataFrame:
    df = pd.read_parquet("../personalCB/sheet9_with_thin_flag.parquet")
    print(f"전체 데이터 로드: {len(df):,}행")
    return df


def prepare_features(df: pd.DataFrame, feature_cols: list, target: str, id_col: str = "ID"):
    """결측 제거 + 씬파일러 정의필드 오염 방지 체크 후 X, y, groups 반환"""
    leak = [c for c in feature_cols if c in THIN_FIELDS]
    if leak:
        raise ValueError(f"씬파일러 정의 필드가 피처에 섞여있습니다: {leak}")

    data = df[feature_cols + [target, id_col]].dropna()
    X = data[feature_cols]
    y = data[target].astype(int)
    groups = data[id_col]
    return X, y, groups


c:\Users\tehun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ------------------------------------------------------------------
# Part 1. 씬파일러군 내부 — 기준모델 vs 대안모델
# ------------------------------------------------------------------
def run_baseline_vs_alt(df: pd.DataFrame, target: str = TARGET):
    print(f"\n{'='*60}\nPart 1. 씬파일러군 기준모델 vs 대안모델 (target={target})\n{'='*60}")

    thin = df[df["is_thin"]].copy()
    print(f"씬파일러군 표본 수: {len(thin):,}")

    results = {}
    for name, feats in [
        ("기준모델(Baseline)", BASELINE_FEATURES),
        ("대안모델(Alternative)", BASELINE_FEATURES + ALT_FEATURES),
    ]:
        X, y, groups = prepare_features(thin, feats, target)
        print(f"\n[{name}] 사용변수: {feats}, 표본수: {len(X):,}, "
              f"양성비율: {y.mean()*100:.2f}%")

        if y.nunique() < 2:
            print("⚠ 양성 클래스가 없어 학습 불가 (표본을 더 확장해야 함)")
            continue

        gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_STATE)
        train_idx, test_idx = next(gss.split(X, y, groups=groups))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # 로지스틱 회귀 (해석 가능한 기본 모델)
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        logit = LogisticRegression(max_iter=1000, class_weight="balanced")
        logit.fit(X_train_s, y_train)
        pred_logit = logit.predict_proba(X_test_s)[:, 1]

        auc_logit = roc_auc_score(y_test, pred_logit)
        pr_auc_logit = average_precision_score(y_test, pred_logit)

        print(f"  [로지스틱회귀] AUC={auc_logit:.4f}, PR-AUC={pr_auc_logit:.4f}")

        # XGBoost (성능 비교용)
        xgb_model = xgb.XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
            eval_metric="aucpr", random_state=RANDOM_STATE,
        )
        xgb_model.fit(X_train, y_train)
        pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

        auc_xgb = roc_auc_score(y_test, pred_xgb)
        pr_auc_xgb = average_precision_score(y_test, pred_xgb)
        print(f"  [XGBoost]     AUC={auc_xgb:.4f}, PR-AUC={pr_auc_xgb:.4f}")

        results[name] = {
            "logit_auc": auc_logit, "logit_prauc": pr_auc_logit,
            "xgb_auc": auc_xgb, "xgb_prauc": pr_auc_xgb,
            "xgb_model": xgb_model, "X_test": X_test, "y_test": y_test,
        }

    if "기준모델(Baseline)" in results and "대안모델(Alternative)" in results:
        gain_auc = results["대안모델(Alternative)"]["xgb_auc"] - results["기준모델(Baseline)"]["xgb_auc"]
        gain_prauc = results["대안모델(Alternative)"]["xgb_prauc"] - results["기준모델(Baseline)"]["xgb_prauc"]
        print(f"\n>>> 대안정보 추가 개선폭 (XGBoost 기준): "
              f"AUC {gain_auc:+.4f}, PR-AUC {gain_prauc:+.4f}")
        print(">>> 이 개선폭이 프로젝트의 핵심 주장(대안정보가 씬파일러 "
              "예측력을 높인다)의 정량적 근거가 됩니다.")

    return results


In [4]:
# ------------------------------------------------------------------
# Part 2. 전체 모집단 — H7 (대체 vs 보완) 검증
# ------------------------------------------------------------------
def run_h7_full_population(df: pd.DataFrame, target: str = TARGET):
    print(f"\n{'='*60}\nPart 2. 전체 모집단 H7 검증 (target={target})\n{'='*60}")

    data = df.copy()
    data["thin_dummy"] = data["is_thin"].astype(int)

    # 대안변수 × 씬파일러더미 교호항 생성
    for alt in ALT_FEATURES:
        data[f"{alt}_x_thin"] = data[alt] * data["thin_dummy"]

    # 모델 1: SCORE + 씬파일러더미만 (대안변수 추가 전)
    feats_before = ["SCORE", "thin_dummy"]
    # 모델 2: 대안변수 + 교호항 추가 (대안변수 추가 후)
    feats_after = feats_before + ALT_FEATURES + [f"{a}_x_thin" for a in ALT_FEATURES]

    coef_summary = {}
    for name, feats in [("추가 전", feats_before), ("추가 후", feats_after)]:
        X, y, _ = prepare_features(data, feats, target)
        if y.nunique() < 2:
            print(f"[{name}] 양성 클래스 부족으로 스킵")
            continue

        scaler = StandardScaler()
        X_s = scaler.fit_transform(X)
        logit = LogisticRegression(max_iter=1000, class_weight="balanced")
        logit.fit(X_s, y)

        thin_idx = feats.index("thin_dummy")
        coef_summary[name] = logit.coef_[0][thin_idx]
        print(f"[{name}] thin_dummy 계수: {coef_summary[name]:.4f} "
              f"(표본수 {len(X):,}, 사용변수 {feats})")

    if "추가 전" in coef_summary and "추가 후" in coef_summary:
        before, after = coef_summary["추가 전"], coef_summary["추가 후"]
        change_pct = (before - after) / abs(before) * 100 if before != 0 else np.nan
        print(f"\n>>> thin_dummy 계수 변화율: {change_pct:.1f}%")
        if abs(change_pct) > 30:
            print(">>> 계수가 크게 감소 → 대안정보가 씬파일러 리스크의 상당 "
                  "부분을 '설명'해냄 → H7 지지 (보완적 성격)")
        else:
            print(">>> 계수가 안정적으로 유지 → 대안정보만으로는 씬파일러 "
                  "고유 리스크를 설명하지 못함 → 추가 대안데이터 필요 시사")



In [5]:
# ------------------------------------------------------------------
# Part 3. SHAP 기여도 분석 (씬파일러군 대안모델 기준)
# ------------------------------------------------------------------
def run_shap_analysis(df: pd.DataFrame, target: str = TARGET):
    print(f"\n{'='*60}\nPart 3. SHAP 기여도 분석 (씬파일러군, target={target})\n{'='*60}")

    thin = df[df["is_thin"]].copy()
    feats = BASELINE_FEATURES + ALT_FEATURES
    X, y, _ = prepare_features(thin, feats, target)

    if y.nunique() < 2:
        print("⚠ 양성 클래스 부족으로 SHAP 분석 스킵")
        return

    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        scale_pos_weight=(y == 0).sum() / max((y == 1).sum(), 1),
        random_state=RANDOM_STATE,
    )
    model.fit(X, y)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    importance = pd.DataFrame({
        "변수": feats,
        "평균절대SHAP값": np.abs(shap_values).mean(axis=0),
    }).sort_values("평균절대SHAP값", ascending=False)

    print("\n변수 중요도 순위 (은행/보증기관 우선 확보 변수 근거):")
    print(importance.to_string(index=False))

    # summary_plot은 스크립트 환경에서는 저장 방식으로 사용
    import matplotlib.pyplot as plt
    shap.summary_plot(shap_values, X, show=False)
    plt.tight_layout()
    plt.savefig("outputs/shap_summary_thin_filer.png", dpi=150)
    print("\n✅ SHAP summary plot 저장: outputs/shap_summary_thin_filer.png")



In [6]:
# ------------------------------------------------------------------
# 실행부
# ------------------------------------------------------------------
if __name__ == "__main__":
    df = load_data()

    run_baseline_vs_alt(df, target=TARGET)
    run_h7_full_population(df, target=TARGET)
    # run_shap_analysis(df, target=TARGET)

    print("\n모든 모델링 단계 완료.")
    print("다른 목표변수(PERF2~4)로도 동일 파이프라인을 반복해 "
          "결과의 일관성을 확인하는 것을 권장합니다.")

전체 데이터 로드: 12,516,144행

Part 1. 씬파일러군 기준모델 vs 대안모델 (target=PERF1)
씬파일러군 표본 수: 3,254,230

[기준모델(Baseline)] 사용변수: ['SCORE', 'SCORE_6M'], 표본수: 3,254,230, 양성비율: 0.10%
  [로지스틱회귀] AUC=0.8869, PR-AUC=0.0072
  [XGBoost]     AUC=0.9049, PR-AUC=0.0177

[대안모델(Alternative)] 사용변수: ['SCORE', 'SCORE_6M', 'AP0910001', 'AP0910002', 'AL012G005', 'AL012G011', 'AL012G019'], 표본수: 3,254,230, 양성비율: 0.10%
  [로지스틱회귀] AUC=0.9110, PR-AUC=0.0116
  [XGBoost]     AUC=0.9360, PR-AUC=0.0242

>>> 대안정보 추가 개선폭 (XGBoost 기준): AUC +0.0311, PR-AUC +0.0065
>>> 이 개선폭이 프로젝트의 핵심 주장(대안정보가 씬파일러 예측력을 높인다)의 정량적 근거가 됩니다.

Part 2. 전체 모집단 H7 검증 (target=PERF1)
[추가 전] thin_dummy 계수: -1.5738 (표본수 12,516,144, 사용변수 ['SCORE', 'thin_dummy'])
[추가 후] thin_dummy 계수: -0.4919 (표본수 12,516,144, 사용변수 ['SCORE', 'thin_dummy', 'AP0910001', 'AP0910002', 'AL012G005', 'AL012G011', 'AL012G019', 'AP0910001_x_thin', 'AP0910002_x_thin', 'AL012G005_x_thin', 'AL012G011_x_thin', 'AL012G019_x_thin'])

>>> thin_dummy 계수 변화율: -68.7%
>>> 계수가 크게 감소 → 대안정보가 씬파일러 리스크의 